# Data problem 1: the file year is not the publication year

Notebook for the video. It only re-counts, on the raw Portal BASE files, a problem we found during the data analysis (`notebooks/00_qualidade.ipynb`, `notebooks/02_anos_treino.ipynb`, commit `b249110`). It takes no decisions.

- **What we assumed:** the 2025 file holds the contracts published in 2025.
- **Why it matters:** we split train, validation and test by publication date (`dataPublicacao`).
- **Question:** in which files are the contracts published in 2025?

It counts rows of every file, including Nov–Dec 2025 (the test period), but only how many there are per file. Like `00_qualidade.ipynb`, that is a technical count, not a look at the test results.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cpv_data.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import cpv_data as cd

def fmt(n):
    return f"{n:,}"

# Every file, 2012 to 2026, keeping only the rows published in 2025
pub_2025 = cd.load_contracts(["dataCelebracaoContrato"],
                             since=pd.Timestamp("2025-01-01"), until=pd.Timestamp("2026-01-01"))
print("Rows published in 2025, read from every file:", fmt(len(pub_2025)))

Rows published in 2025, read from every file: 255,011


## In which file is each contract published in 2025?

In [2]:
por_ficheiro = pub_2025["ficheiro"].value_counts().sort_index().rename("published in 2025").to_frame()
por_ficheiro["share"] = por_ficheiro["published in 2025"] / len(pub_2025)
por_ficheiro = por_ficheiro[por_ficheiro["published in 2025"] > 0]
por_ficheiro.index.name = "file"
por_ficheiro.style.format({"published in 2025": "{:,}", "share": "{:.1%}"})

,published in 2025,share
file,,
2014,2,0.0%
2015,4,0.0%
2016,6,0.0%
2017,11,0.0%
2018,5,0.0%
2019,24,0.0%
2020,43,0.0%
2021,249,0.1%
2022,747,0.3%


## So what is the year of a file?

Take the contracts published in 2025 that sit in the 2024 file, and look at the year they were signed (`dataCelebracaoContrato`).

In [3]:
no_2024 = pub_2025[pub_2025["ficheiro"] == 2024]
assinado = pd.to_datetime(no_2024["dataCelebracaoContrato"], errors="coerce").dt.year.astype("Int64")
print("Contracts published in 2025 that sit in the 2024 file:", fmt(len(no_2024)))
print("   of which signed in 2024:", fmt(int((assinado == 2024).sum())))
assinado.value_counts(dropna=False).sort_index().rename("contracts").rename_axis("year signed").to_frame()

Contracts published in 2025 that sit in the 2024 file: 25,877
   of which signed in 2024: 25,777


,contracts
year signed,
2024,25777
<NA>,100


## What our first measurements missed

The first quality checks read only the 2025 and 2026 files.

In [4]:
fora = ~pub_2025["ficheiro"].isin([2025, 2026])
print("Published in 2025 but outside the 2025 and 2026 files:",
      fmt(int(fora.sum())), f"({fora.mean():.1%})")

Published in 2025 but outside the 2025 and 2026 files: 29,898 (11.7%)


In [5]:
# Where the missed contracts would fall in our split (train Jan–Sep, validation Oct, test Nov–Dec 2025)
nomes = {"treino": "train (Jan–Sep)", "validação": "validation (Oct)", "teste": "test (Nov–Dec)"}
parte = cd.period(pub_2025["data"]).map(nomes)
tabela = pd.DataFrame({
    "all published in 2025": parte.value_counts(),
    "missed (outside 2025/2026 files)": parte[fora].value_counts(),
}).reindex(list(nomes.values()))
tabela["share missed"] = tabela["missed (outside 2025/2026 files)"] / tabela["all published in 2025"]
tabela.style.format({"all published in 2025": "{:,.0f}", "missed (outside 2025/2026 files)": "{:,.0f}", "share missed": "{:.1%}"})

,all published in 2025,missed (outside 2025/2026 files),share missed
train (Jan–Sep),"193,638","27,283",14.1%
validation (Oct),"23,980","1,336",5.6%
test (Nov–Dec),"37,393","1,279",3.4%


## How we fixed it

- The file year is the **signing** year. Contracts published in a year are spread over several files.
- `cd.load_contracts` reads **every** file by default and filters by `dataPublicacao`, file by file (to save memory). The split into train, validation and test uses only `dataPublicacao`.
- The measurements taken with only the 2025 and 2026 files were redone with every file (PR #4).

**Lesson that applies to any project:** check what a column, or the way files are split, really means, with a count. A file name is not documentation.